# Pipeline A1 — Frozen-prior GeoWeighting

Train three new shared models from scratch with the same four anchor forwards as CE+KD baseline, changing only mean-one loss weights derived from cached baseline geometry. This notebook reads baseline artifacts from `/kaggle/input/datasets/dyhngg/checkpoint-new-prune`; it does not rerun baseline or specialized models.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU'
print('Detected GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(f'GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); env.pop('GITHUB_TOKEN_RUNTIME', None); github_token = None
assert (PROJECT_ROOT / 'scripts' / 'run_geoweighting.py').is_file(), 'Commit and push A1 files first'
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1', 'tabulate>=0.9'], check=True)

## Resolve and validate cached baseline

In [ ]:
from datetime import datetime, timezone
import numpy as np, pandas as pd, yaml
from baseline_artifacts import find_confirmatory_root, compute_anchor_geometry_prior
INPUT_ROOT = Path('/kaggle/input/datasets/dyhngg/checkpoint-new-prune')
BASELINE_ROOT = find_confirmatory_root(INPUT_ROOT)
print('Resolved baseline:', BASELINE_ROOT)
config = yaml.safe_load((PROJECT_ROOT / 'configs' / 'kaggle_geoweighting_a1.yaml').read_text())
assert Path(config['baseline_artifacts']['input_root']) == INPUT_ROOT
assert config['experiment']['seeds'] == [0, 1, 2] and config['training']['epochs'] == 20
assert config['compression']['train_widths'] == [0.25, 0.50, 0.75, 1.00]
assert config['geometry_weighting'] == {'alpha': 1.0, 'beta': 0.5, 'epsilon': 1e-8}
prior, anchor_weights = compute_anchor_geometry_prior(BASELINE_ROOT, config['compression']['train_widths'], **config['geometry_weighting'])
assert np.isclose(np.mean(list(anchor_weights.values())), 1.0) and all(w > 0 for w in anchor_weights.values())
display(prior); print('Frozen anchor weights:', anchor_weights)
RUN_NAME = datetime.now(timezone.utc).strftime('kaggle-geoweighting-a1-%Y%m%d-%H%M%S')
RUN_DIR = Path('/kaggle/working/new-pruning-outputs') / RUN_NAME
config['experiment']['output_dir'] = str(RUN_DIR)
RESOLVED_CONFIG = Path('/kaggle/working/kaggle_geoweighting_a1_resolved.yaml')
RESOLVED_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))

## Train three GeoWeighting shared models

In [ ]:
import time
from scripts.run_geoweighting import run, finalize_geoweighting
from scripts.run_multi_gpu import run_multi_gpu
started = time.perf_counter()
if torch.cuda.device_count() >= 2:
    report_path = run_multi_gpu(RESOLVED_CONFIG, [0, 1], runner_module='scripts.run_geoweighting', finalize=finalize_geoweighting)
else:
    report_path = run(RESOLVED_CONFIG)
print(f'Completed in {(time.perf_counter()-started)/3600:.2f} hours')
print('Base report:', report_path)

## Paired baseline-versus-A1 evaluation

In [ ]:
baseline = pd.read_csv(BASELINE_ROOT / 'central_analysis_all_seeds.csv')
a1 = pd.read_csv(RUN_DIR / 'central_analysis_all_seeds.csv')
anchors = {0.25, 0.50, 0.75, 1.00}
def summarize(frame, method):
    rows = []
    for seed, seed_frame in frame.groupby('seed'):
        seed_frame = seed_frame.sort_values('budget')
        unseen = seed_frame.loc[~seed_frame['budget'].round(2).isin(anchors)]
        local = seed_frame.loc[seed_frame['local_wasserstein_sensitivity'].notna()]
        cliff = local.loc[local['budget'].between(0.30, 0.45)]
        anchor_rows = seed_frame.loc[seed_frame['budget'].round(2).isin(anchors)]
        x = np.log(seed_frame['flops'].to_numpy(float)); y = seed_frame['accuracy'].to_numpy(float)
        rows.append({'method': method, 'seed': int(seed), 'mean_unseen_accuracy': unseen['accuracy'].mean(), 'worst_unseen_accuracy': unseen['accuracy'].min(), 'mean_G': local['local_wasserstein_sensitivity'].mean(), 'max_G': local['local_wasserstein_sensitivity'].max(), 'cliff_region_max_G': cliff['local_wasserstein_sensitivity'].max(), 'mean_anchor_accuracy': anchor_rows['accuracy'].mean(), 'full_width_accuracy': seed_frame.loc[np.isclose(seed_frame['budget'], 1.0), 'accuracy'].iloc[0], 'accuracy_logflops_auc': np.trapz(y, x)/(x[-1]-x[0])})
    return pd.DataFrame(rows)
summary = pd.concat([summarize(baseline, 'baseline'), summarize(a1, 'GeoWeighting A1')], ignore_index=True)
wide = summary.pivot(index='seed', columns='method')
comparison = pd.DataFrame({'seed': wide.index})
for metric in ['mean_unseen_accuracy','worst_unseen_accuracy','mean_G','max_G','cliff_region_max_G','mean_anchor_accuracy','full_width_accuracy','accuracy_logflops_auc']:
    comparison[f'baseline_{metric}'] = wide[metric]['baseline'].to_numpy()
    comparison[f'a1_{metric}'] = wide[metric]['GeoWeighting A1'].to_numpy()
    comparison[f'delta_{metric}'] = comparison[f'a1_{metric}'] - comparison[f'baseline_{metric}']
summary.to_csv(RUN_DIR / 'baseline_a1_metrics_by_seed.csv', index=False)
comparison.to_csv(RUN_DIR / 'baseline_a1_paired_comparison.csv', index=False)
display(comparison)

In [ ]:
mean_unseen_pass = comparison['delta_mean_unseen_accuracy'].mean() > 0 and (comparison['delta_mean_unseen_accuracy'] > 0).sum() >= 2
worst_pass = comparison['delta_worst_unseen_accuracy'].mean() >= -0.002 and comparison['delta_worst_unseen_accuracy'].min() >= -0.005
geometry_pass = comparison['delta_max_G'].mean() < 0 or comparison['delta_cliff_region_max_G'].mean() < 0
anchor_pass = comparison['delta_mean_anchor_accuracy'].mean() >= -0.003 and comparison['delta_full_width_accuracy'].mean() >= -0.003
A1_GO = bool(mean_unseen_pass and worst_pass and geometry_pass and anchor_pass)
decision = 'A1 GO — run specialized A2' if A1_GO else 'A1 NO-GO / inspect trade-offs before A2'
from IPython.display import Markdown, display
display(Markdown(f'## {decision}'))
print({'mean_unseen_pass': mean_unseen_pass, 'worst_pass': worst_pass, 'geometry_pass': geometry_pass, 'anchor_pass': anchor_pass})

In [ ]:
import matplotlib.pyplot as plt
PLOT_DIR = RUN_DIR / 'plots'; PLOT_DIR.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for method, frame in [('Baseline', baseline), ('GeoWeighting A1', a1)]:
    mean = frame.groupby('budget')['accuracy'].mean(); axes[0].plot(mean.index, mean, marker='o', label=method)
    local = frame.dropna(subset=['local_wasserstein_sensitivity']).groupby('budget')['local_wasserstein_sensitivity'].mean(); axes[1].plot(local.index, local, marker='o', label=method)
axes[0].set(xlabel='Width', ylabel='Mean test accuracy', title='Dense-budget generalization')
axes[1].set(xlabel='Interval start c', ylabel='Mean G(c)', title='Local geometric instability')
for ax in axes: ax.grid(alpha=.25); ax.legend()
fig.tight_layout(); fig.savefig(PLOT_DIR/'baseline_vs_a1.png', dpi=180); plt.show()

In [ ]:
report_lines = ['# GeoWeighting A1 report','',f'Decision: **{decision}**','', '## Frozen prior','',prior.to_markdown(index=False),'','## Paired comparison','',comparison.to_markdown(index=False),'', '> A1 trains new models from scratch. Baseline artifacts are reused only as a frozen geometry prior and comparison reference. No specialized models are trained in A1.']
REPORT = RUN_DIR/'reports'/'geoweighting_a1_report.md'; REPORT.parent.mkdir(parents=True, exist_ok=True); REPORT.write_text('\n'.join(report_lines)+'\n')
display(Markdown('\n'.join(report_lines)))

In [ ]:
import shutil, zipfile
from IPython.display import FileLink
files = sorted(p for p in RUN_DIR.rglob('*') if p.is_file())
pd.DataFrame({'relative_path':[str(p.relative_to(RUN_DIR)) for p in files],'size_bytes':[p.stat().st_size for p in files]}).to_csv(RUN_DIR/'artifact_manifest.csv',index=False)
archive = Path(shutil.make_archive(str(Path('/kaggle/working')/RUN_NAME),'zip',root_dir=RUN_DIR))
with zipfile.ZipFile(archive) as zf: names=set(zf.namelist())
for required in ['frozen_anchor_geometry_prior.csv','baseline_a1_paired_comparison.csv','reports/geoweighting_a1_report.md','seed_0/checkpoint.pt','seed_1/checkpoint.pt','seed_2/checkpoint.pt']: assert required in names
print('Archive:',archive); display(FileLink(str(archive))); print('Save Version after completion.')